In [2]:
import pandas as pd

df1 = pd.read_csv("data/phishing1.csv")

df1.shape


(48812, 3)

In [3]:
df1.head()


,url,label,source
0,http://110.37.26.193:54956/bin.sh,1,urlhaus
1,https://sentry5.obor1shwron8.ru/4ylkytvt,1,urlhaus
2,https://d6gu.ripple-cask.ru/jid43vpd,1,urlhaus
3,http://130.12.180.34/596a96cc7bf9108cd896f33c4...,1,urlhaus
4,https://bronze.systems,0,tranco


In [4]:
df1.columns

Index(['url', 'label', 'source'], dtype='str')

In [5]:
df1['label'].unique()

array([1, 0])

In [6]:
df1.isnull().sum()

url       0
label     0
source    0
dtype: int64

In [7]:
df1.duplicated().sum()

np.int64(0)

In [8]:
df1 = df1[['url', 'label']]
df1.head()

,url,label
0,http://110.37.26.193:54956/bin.sh,1
1,https://sentry5.obor1shwron8.ru/4ylkytvt,1
2,https://d6gu.ripple-cask.ru/jid43vpd,1
3,http://130.12.180.34/596a96cc7bf9108cd896f33c4...,1
4,https://bronze.systems,0


In [9]:
df2 = pd.read_csv("data/phishing2.csv")

df2.shape

(651191, 68)

In [10]:
df2.columns

Index(['url', 'type', 'label', 'web_is_live', 'web_security_score',
       'web_forms_count', 'web_password_fields', 'web_has_login',
       'web_ssl_valid', 'url_len', '@', '?', '-', '=', '.', '#', '%', '+', '$',
       '!', '*', ',', '//', 'digits', 'letters', 'domain', 'abnormal_url',
       'https', 'Shortining_Service', 'having_ip_address',
       'defac_has_hacked_terms', 'defac_has_suspicious_ext',
       'defac_path_depth', 'defac_is_deep_path', 'defac_path_underscores',
       'defac_is_gov_edu', 'defac_has_index_php', 'defac_has_option_param',
       'phish_has_brand', 'phish_brand_in_subdomain', 'phish_brand_in_path',
       'phish_hyphen_count', 'phish_digit_count', 'phish_long_domain',
       'phish_many_subdomains', 'phish_suspicious_tld', 'phish_keyword_count',
       'phish_has_redirect', 'phish_param_count', 'phish_encoded_chars',
       'enh_urgency_count', 'enh_security_count', 'enh_brand_count',
       'enh_brand_hijack', 'enh_subdomain_count', 'enh_long_path',
    

In [11]:
df2['label'].unique()

array([ 2.,  0.,  1.,  3., nan])

In [12]:
df2 = df2[df2['label'].isin([0, 2])]
df2['label'].unique()

array([2., 0.])

In [13]:
df2['label'] = df2['label'].replace({2: 1})

df2['label'].unique()

array([1., 0.])

In [14]:
df2 = df2[['url', 'label']]

df2.head()

,url,label
0,br-icloud.com.br,1.0
1,mp3raid.com/music/krizz_kaliko.html,0.0
2,bopsecrets.org/rexroth/cr/1.htm,0.0
5,http://buzzfil.net/m/show-art/ils-etaient-loin...,0.0
6,espn.go.com/nba/player/_/id/3457/brandon-rush,0.0


In [15]:
df2.shape

(522142, 2)

In [16]:
import re
import numpy as np
from urllib.parse import urlparse
from scipy.stats import entropy

def extract_url_features(url):
    features = {}
    
    features['url_length'] = len(url)
    
    features['has_https'] = 1 if url.startswith("https") else 0
    
    features['dot_count'] = url.count('.')
    
    return features

In [17]:
extract_url_features(df1['url'].iloc[0])

{'url_length': 33, 'has_https': 0, 'dot_count': 4}

In [18]:
def extract_url_features(url):
    features = {}
    
    features['url_length'] = len(url)
    
    features['has_https'] = 1 if url.startswith("https") else 0
    
    features['dot_count'] = url.count('.')
    
    features['has_ip'] = 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0
    
    features['special_char_count'] = len(re.findall(r'[^\w]', url))
    
    features['digit_count'] = sum(c.isdigit() for c in url)
    
    return features

In [19]:
extract_url_features(df1['url'].iloc[0])

{'url_length': 33,
 'has_https': 0,
 'dot_count': 4,
 'has_ip': 1,
 'special_char_count': 9,
 'digit_count': 15}

In [20]:
def extract_url_features(url):
    features = {}
    
    features['url_length'] = len(url)
    
    features['has_https'] = 1 if url.startswith("https") else 0
    
    features['dot_count'] = url.count('.')
    
    features['has_ip'] = 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0
    
    features['special_char_count'] = len(re.findall(r'[^\w]', url))
    
    features['digit_count'] = sum(c.isdigit() for c in url)
    
    prob = [float(url.count(c)) / len(url) for c in dict.fromkeys(list(url))]
    features['url_entropy'] = entropy(prob)
    
    return features

In [21]:
extract_url_features(df1['url'].iloc[0])

{'url_length': 33,
 'has_https': 0,
 'dot_count': 4,
 'has_ip': 1,
 'special_char_count': 9,
 'digit_count': 15,
 'url_entropy': np.float64(2.834661752244436)}

In [22]:
feature_list = df1['url'].apply(extract_url_features)

features_df1 = pd.DataFrame(feature_list.tolist())

features_df1.head()

,url_length,has_https,dot_count,has_ip,special_char_count,digit_count,url_entropy
0,33,0,4,1,9,15,2.834662
1,40,1,2,0,6,4,2.915515
2,36,1,2,0,7,3,3.015335
3,91,0,4,1,9,46,3.020555
4,22,1,1,0,4,0,2.563151


In [23]:
features_df1['label'] = df1['label'].values

features_df1.head()

,url_length,has_https,dot_count,has_ip,special_char_count,digit_count,url_entropy,label
0,33,0,4,1,9,15,2.834662,1
1,40,1,2,0,6,4,2.915515,1
2,36,1,2,0,7,3,3.015335,1
3,91,0,4,1,9,46,3.020555,1
4,22,1,1,0,4,0,2.563151,0


In [24]:
features_df1.shape

(48812, 8)

In [25]:
X = features_df1.drop('label', axis=1)
y = features_df1['label']

X.shape, y.shape

((48812, 7), (48812,))

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((39049, 7), (9763, 7))

In [27]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9980538768821059
Precision: 0.998359983599836
Recall: 0.9977463634501127
F1-score: 0.9980530792089354
ROC-AUC: 0.9995561718343524

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4882
           1       1.00      1.00      1.00      4881

    accuracy                           1.00      9763
   macro avg       1.00      1.00      1.00      9763
weighted avg       1.00      1.00      1.00      9763



In [37]:
feature_list_df2 = df2['url'].apply(extract_url_features)
features_df2 = pd.DataFrame(feature_list_df2.tolist())

features_df2['label'] = df2['label'].values

features_df2.head()


,url_length,has_https,dot_count,has_ip,special_char_count,digit_count,url_entropy,label
0,16,0,2,0,3,0,2.339372,1.0
1,35,0,2,0,4,1,2.827447,0.0
2,31,0,2,0,5,1,2.570254,0.0
3,118,0,2,0,24,1,2.939321,0.0
4,45,0,2,0,9,4,3.039973,0.0


In [38]:
features_df2.shape

(522142, 8)

In [39]:
X_external = features_df2.drop('label', axis=1)
y_external = features_df2['label']

In [40]:
y_external_pred = model.predict(X_external)
y_external_proba = model.predict_proba(X_external)[:,1]

In [41]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("External Accuracy:", accuracy_score(y_external, y_external_pred))
print("External Precision:", precision_score(y_external, y_external_pred))
print("External Recall:", recall_score(y_external, y_external_pred))
print("External F1:", f1_score(y_external, y_external_pred))
print("External ROC-AUC:", roc_auc_score(y_external, y_external_proba))

External Accuracy: 0.17829249514499887
External Precision: 0.17824491465374712
External Recall: 0.9881830666538916
External F1: 0.3020136848578317
External ROC-AUC: 0.5952366609700618


In [42]:
features_df2['label'].value_counts()

label
0.0    428209
1.0     93933
Name: count, dtype: int64